In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/syahribanun/fp-nlp/dataset_dedup.jsonl
/kaggle/input/datasets/syahribanun/fp-nlp/holdout.jsonl


# Install

In [8]:
!pip install -q -U transformers accelerate
!pip install -q antlr4-python3-runtime==4.11   # utk sympy parse_latex

# Config

In [9]:
import json, glob, os, gc, torch
hits = glob.glob('/kaggle/input/**/holdout.jsonl', recursive=True)
HOLDOUT = hits[0] if hits else '/kaggle/input/holdout.jsonl'
OUT = '/kaggle/working/skenario4_results.json'
MAX_NEW_TOKENS = 4096   # reasoning model (OpenMath) butuh panjang; lebih lambat
BATCH = 16
MODELS = {
    'OpenMath-Nemotron-1.5B': 'nvidia/OpenMath-Nemotron-1.5B',
    'SeaLLMs-v3-1.5B-Chat'  : 'SeaLLMs/SeaLLMs-v3-1.5B-Chat',
    'Qwen2.5-1.5B-Instruct' : 'Qwen/Qwen2.5-1.5B-Instruct',
}
print('HOLDOUT =', HOLDOUT)

HOLDOUT = /kaggle/input/datasets/syahribanun/fp-nlp/holdout.jsonl


# Cek jawaban

In [10]:
import re
from fractions import Fraction
try:
    import sympy
    from sympy.parsing.latex import parse_latex
    from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                            implicit_multiplication_application)
    _T = standard_transformations + (implicit_multiplication_application,)
    _SYMPY = True
except Exception:
    _SYMPY = False
BS = chr(92)

def extract_boxed(text):
    if not text: return None
    m = BS + 'boxed'; idx = text.rfind(m)
    if idx == -1: return None
    i = idx + len(m)
    while i < len(text) and text[i] != '{':
        if not text[i].isspace(): return None
        i += 1
    if i >= len(text): return None
    d=0; out=[]
    for ch in text[i:]:
        if ch=='{':
            d+=1
            if d==1: continue
        elif ch=='}':
            d-=1
            if d==0: return ''.join(out)
        out.append(ch)
    return None

def norm(s):
    if s is None: return ''
    s = str(s).strip().strip('$').strip().replace(' ','')
    s = re.sub(r'(?<=\d),(?=\d{3}\b)','',s)
    s = re.sub(r'(?<=\d),(?=\d)','.',s)
    return s.rstrip('.').lower()

def _expr(s):
    if not _SYMPY: return None
    s=s.strip().strip('$').strip()
    if re.fullmatch(r'-?\d+/\d+',s):
        try: return sympy.Rational(s)
        except Exception: pass
    for p in ('latex','plain'):
        try:
            return parse_latex(s) if p=='latex' else parse_expr(s.replace('^','**'), transformations=_T)
        except Exception: continue
    return None

def _num_eq(a,b,tol=1e-6):
    def f(x):
        x=x.rstrip('%')
        try: return float(Fraction(x))
        except Exception:
            try: return float(x)
            except Exception: return None
    na,nb=f(a),f(b)
    return na is not None and nb is not None and abs(na-nb)<=tol*max(1.0,abs(nb))

def is_correct(pred, gold):
    if pred is None: return False
    pn, gn = norm(pred), norm(gold)
    if not gn: return False
    if pn==gn: return True
    if _num_eq(pn,gn): return True
    if _SYMPY:
        ea,eb=_expr(pred),_expr(gold)
        if ea is not None and eb is not None:
            try: return sympy.simplify(ea-eb)==0
            except Exception: return False
    return False

def extract_answer(gen):
    b = extract_boxed(gen)
    if b is not None: return b
    nums = re.findall(r'-?\d+(?:[.,]\d+)?', gen or '')
    return nums[-1] if nums else None

def grade(gen, gold):
    b = extract_boxed(gen)
    p = b if b is not None else extract_answer(gen)
    return {'pred':p, 'has_boxed':b is not None, 'correct':is_correct(p,gold)}

assert grade('x '+BS+'boxed{42}','42')['correct']
print('checker OK')

checker OK


# Load Holdout

In [11]:
rows = [json.loads(l) for l in open(HOLDOUT,encoding='utf-8') if l.strip()]
PROMPT = ('Selesaikan soal matematika berikut. Tunjukkan langkah-langkah '
          'penyelesaian secara rinci. Pastikan jawaban akhir berada di dalam '
          + BS + 'boxed{}.' + chr(10)+chr(10))
print('holdout:', len(rows), 'soal')

holdout: 300 soal


# Eval Model

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
results = {}
for name, mid in MODELS.items():
    print('==='*8, name)
    tok = AutoTokenizer.from_pretrained(mid)
    tok.padding_side='left'
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=torch.float16,
                                                 device_map='auto', attn_implementation='sdpa')
    model.eval()
    gens=[]
    for i in range(0,len(rows),BATCH):
        batch=rows[i:i+BATCH]
        prompts=[tok.apply_chat_template([{'role':'user','content':PROMPT+r['soal']}],
                 tokenize=False, add_generation_prompt=True) for r in batch]
        enc=tok(prompts,return_tensors='pt',padding=True,truncation=True,max_length=2048).to(0)
        with torch.no_grad():
            out=model.generate(**enc,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,
                               pad_token_id=tok.pad_token_id)
        gens+=tok.batch_decode(out[:,enc.input_ids.shape[1]:],skip_special_tokens=True)
        print(f'  {min(i+BATCH,len(rows))}/{len(rows)}')
    correct=sum(grade(g,r['jawaban'])['correct'] for r,g in zip(rows,gens))
    boxed=sum(grade(g,r['jawaban'])['has_boxed'] for r,g in zip(rows,gens))
    results[name]={'acc':round(correct/len(rows),4),'format_ok':round(boxed/len(rows),4),
                   'correct':correct,'n':len(rows)}
    json.dump({'gens':gens}, open(f'/kaggle/working/gen_{name}.json','w',encoding='utf-8'), ensure_ascii=False)
    print('  ->', results[name])
    del model; gc.collect(); torch.cuda.empty_cache()

======================== OpenMath-Nemotron-1.5B


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  16/300
  32/300
  48/300
  64/300
  80/300
  96/300
  112/300
  128/300
  144/300
  160/300
  176/300
  192/300
  208/300
  224/300
  240/300
  256/300
  272/300
  288/300
  300/300
  -> {'acc': 0.0233, 'format_ok': 0.0833, 'correct': 7, 'n': 300}
======================== SeaLLMs-v3-1.5B-Chat


config.json:   0%|          | 0.00/767 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  16/300
  32/300
  48/300
  64/300
  80/300
  96/300
  112/300
  128/300
  144/300
  160/300
  176/300
  192/300
  208/300
  224/300
  240/300
  256/300
  272/300
  288/300
  300/300
  -> {'acc': 0.0367, 'format_ok': 0.6067, 'correct': 11, 'n': 300}


# Comparison

In [13]:
json.dump(results, open(OUT,'w',encoding='utf-8'), ensure_ascii=False, indent=2)
print(chr(10)+'=== SKENARIO 4 — akurasi di holdout KITA ==='+chr(10))
print(f"{'model':30} {'acc':>7} {'format_ok':>10}")
for n,s in results.items():
    print(f"{n:30} {s['acc']:>7.3f} {s['format_ok']:>10.3f}")
print(chr(10)+'Hasil: skenario4_results.json + gen_*.json di panel Output.')


=== SKENARIO 4 — akurasi di holdout KITA ===

model                              acc  format_ok
OpenMath-Nemotron-1.5B           0.023      0.083
SeaLLMs-v3-1.5B-Chat             0.000      0.010
Qwen2.5-1.5B-Instruct            0.037      0.607

Hasil: skenario4_results.json + gen_*.json di panel Output.
